# M4 — XGBoost with and without class weighting

**Question:** Does multiclass XGBoost, with or without class weighting, improve on the selected M2 Version A Random Forest baseline?

## Experiments

1. `XGB_CLASS_WEIGHT_NONE`: XGBoost trained without sample weights.
2. `XGB_CLASS_WEIGHT_BALANCED`: XGBoost trained with fold-specific balanced per-beat sample weights.

**Important implementation detail:** XGBoost's multiclass objective does not use scikit-learn's `class_weight` argument. The balanced run therefore passes `sample_weight` to the training DMatrix. For each fold, weights are calculated only from that fold's training labels using `n_train / (n_present_classes * count_of_class)`. Validation labels are never used to compute weights. This is the multiclass equivalent of balanced class weighting.

Both runs use the same 17 M2 Version A features and the exact saved M0 person-grouped splits. XGBoost settings are fixed in this notebook; there is no hyperparameter search.

**Primary retention test:** each XGBoost configuration versus the saved M2 Version A OOF baseline on 5-class Macro-F1. Keep only if the paired record-level 95% bootstrap CI lower bound is above zero. The 4-class result is secondary.

> This is a research experiment on MIT-BIH data, not a clinically validated diagnostic model.

**Scope note:** the original M4 plan specified Random Forest plus *one* gradient-boosted family (HistGradientBoosting **or** XGBoost), with/without weighting, for at most four configurations. Running both the already prepared HistGradientBoosting experiment and this XGBoost experiment expands M4 beyond that original four-configuration plan. If you want to preserve the original scope, choose one gradient-boosted family before running both.

## Required Kaggle inputs

**M0/M1 source artifacts**
- `MITBIH_MLII_DualLead_Dataset_corrected.csv`
- `validated_reconstructed_beat_table.csv.gz` (or `.csv`)
- `m0_row_fold_membership.csv.gz` (or `.csv`)
- `m0_person_fold_membership.csv`
- `m0_split_fingerprint.txt`
- `m0_corrected_oof_predictions_REFERENCE_MODEL.csv.gz` (or `.csv`)

**M2 Version A outputs — both required**
- `m2_rr_features_with_sample_positions.csv.gz` (or `.csv`)
- `m2_oof_predictions_REFERENCE_MODEL_PLUS_RR_FEATURES.csv.gz` (or `.csv`)

Attach Version A only, not Version B. The M2 OOF file is mandatory so the XGBoost models are compared to the actual saved M2 baseline rather than to an XGBoost fallback.

## Run instructions

1. First run with `RUN_MODEL_TRAINING = False`. Confirm all data, split, OOF and M2 Version A checks pass.
2. Set `RUN_MODEL_TRAINING = True` only after validation passes.
3. Use **Save Version → Save & Run All (Commit)** in Kaggle.
4. Download `/kaggle/working/ECG_Project/09_M4_XGBOOST_CLASS_WEIGHTING/` after the committed run completes.

The notebook runs both weighting modes over all 25 saved folds. XGBoost uses the CPU `hist` tree method.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import sys
import time

import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)

try:
    import xgboost as xgb
except ImportError as exc:
    raise ImportError(
        "XGBoost is not installed. In a Kaggle notebook, run `%pip install xgboost` "
        "in a separate cell, restart the session if prompted, and run again."
    ) from exc

# Keep False for the first validation-only run. Set True for the full M4 run.
RUN_MODEL_TRAINING = True

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/ECG_Project/09_M4_XGBOOST_CLASS_WEIGHTING")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_ORDER = ["N", "S", "V", "F", "Q"]
CLASS_TO_INT = {name: index for index, name in enumerate(CLASS_ORDER)}
INT_TO_CLASS = {index: name for name, index in CLASS_TO_INT.items()}
FOUR_CLASS_ORDER = ["N", "S", "V", "Q"]
EXPECTED_COUNTS = {"N": 90341, "S": 2781, "V": 7229, "F": 802, "Q": 3894}
EXPECTED_BEATS = 105047
EXPECTED_RECORDS = 46
EXPECTED_PEOPLE = 45

N_SPLITS = 5
N_REPEATS = 5
SEED = 42
NUM_BOOST_ROUND = 200
XGB_PARAMS = {
    "objective": "multi:softprob",
    "num_class": len(CLASS_ORDER),
    "eval_metric": "mlogloss",
    "tree_method": "hist",
    "max_depth": 6,
    "eta": 0.1,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "min_child_weight": 1.0,
    "lambda": 1.0,
    "seed": SEED,
    "nthread": -1,
    "verbosity": 0,
}
BOOTSTRAP_SAMPLES = 1000
BOOTSTRAP_SEED = 20261105
EXPECTED_SPLIT_FINGERPRINT = "5a07f73452ca78af0b212db5e9f060fc4fcc66f49374c2462249189b847440d9"

SIGNAL_COLS = [f"MLII_{i}" for i in range(201)] + [f"Lead2_{i}" for i in range(201)]
AAMI_MAP = {
    "N": "N", "L": "N", "R": "N", "e": "N", "j": "N",
    "A": "S", "a": "S", "J": "S", "S": "S",
    "V": "V", "E": "V", "F": "F", "/": "Q", "f": "Q", "Q": "Q",
}
RR_FEATURES = [
    "pre_RR", "post_RR", "mean_previous_10_RR",
    "pre_RR_div_mean_previous_10_RR", "post_RR_div_pre_RR",
]
PROB_COLS = [f"Prob_{c}" for c in CLASS_ORDER]
EXPECTED_RR_MISSING = {
    "pre_RR": 716,
    "post_RR": 719,
    "mean_previous_10_RR": 259,
    "pre_RR_div_mean_previous_10_RR": 946,
    "post_RR_div_pre_RR": 1385,
}
EXPECTED_ROWS_WITH_MISSING_RR = 1614

BASELINE_NAME = "M2_VERSION_A_REFERENCE_PLUS_RR"
UNWEIGHTED_NAME = "XGB_CLASS_WEIGHT_NONE"
WEIGHTED_NAME = "XGB_CLASS_WEIGHT_BALANCED_SAMPLE_WEIGHT"

print("PHASE M4 — XGBOOST CLASS WEIGHTING")
print("Training enabled:", RUN_MODEL_TRAINING)
print("Output folder:", OUTPUT_DIR)
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__, "| Pandas:", pd.__version__, "| scikit-learn:", sklearn.__version__)
print("XGBoost:", xgb.__version__)
print("XGBoost rounds:", NUM_BOOST_ROUND, "| max_depth:", XGB_PARAMS["max_depth"], "| eta:", XGB_PARAMS["eta"])
print("Weighting modes: none and balanced per-training-beat sample weights")

PHASE M4 — XGBOOST CLASS WEIGHTING
Training enabled: True
Output folder: /kaggle/working/ECG_Project/09_M4_XGBOOST_CLASS_WEIGHTING
Python: 3.13.18
NumPy: 2.1.3 | Pandas: 2.3.3 | scikit-learn: 1.6.1
XGBoost: 3.4.1
XGBoost rounds: 200 | max_depth: 6 | eta: 0.1
Weighting modes: none and balanced per-training-beat sample weights


In [2]:
def find_one(filename_options):
    matches = []
    for filename in filename_options:
        matches.extend(p for p in INPUT_ROOT.rglob(filename) if p.is_file())
    matches = sorted(set(matches))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one matching file from {filename_options}. Found {len(matches)}:\n"
            + "\n".join(str(p) for p in matches)
        )
    print("FOUND:", matches[0])
    return matches[0]


def validate_oof(df, name, aami_df, row_fold):
    required = [
        "Repeat", "Fold", "Row_Index", "Record_ID", "Person_ID",
        "True_Label", "Predicted_Label",
    ] + PROB_COLS
    missing_cols = [col for col in required if col not in df.columns]
    if missing_cols:
        raise ValueError(f"{name}: missing columns {missing_cols}")

    out = df.copy()
    for col in ["Repeat", "Fold", "Row_Index", "Record_ID"]:
        out[col] = pd.to_numeric(out[col], errors="raise").astype(int)
    for col in ["Person_ID", "True_Label", "Predicted_Label"]:
        out[col] = out[col].astype(str).str.strip()

    expected_rows = EXPECTED_BEATS * N_REPEATS
    if len(out) != expected_rows:
        raise ValueError(f"{name}: expected {expected_rows} rows, found {len(out)}.")
    if out.duplicated(["Repeat", "Row_Index"]).any():
        raise ValueError(f"{name}: duplicate (Repeat, Row_Index) pairs.")

    truth = aami_df[["Row_Index", "Record_ID", "Person_ID", "AAMI_Label"]]
    check = out.merge(truth, on="Row_Index", how="left", validate="many_to_one", suffixes=("", "_truth"))
    if check["AAMI_Label"].isna().any():
        raise ValueError(f"{name}: unknown Row_Index values.")
    for left, right in [
        ("Record_ID", "Record_ID_truth"),
        ("Person_ID", "Person_ID_truth"),
        ("True_Label", "AAMI_Label"),
    ]:
        if not (check[left].astype(str) == check[right].astype(str)).all():
            raise ValueError(f"{name}: mismatch in {left}.")

    expected_fold = row_fold[["Row_Index", "Repeat", "Fold"]].rename(columns={"Fold": "Expected_Fold"})
    fold_check = out.merge(expected_fold, on=["Row_Index", "Repeat"], how="left", validate="one_to_one")
    if fold_check["Expected_Fold"].isna().any() or not (fold_check["Fold"] == fold_check["Expected_Fold"]).all():
        raise ValueError(f"{name}: predictions do not use the locked M0 folds.")

    probabilities = out[PROB_COLS].to_numpy(dtype=float)
    if not np.isfinite(probabilities).all():
        raise ValueError(f"{name}: probabilities contain non-finite values.")
    if (probabilities < -1e-7).any() or (probabilities > 1.0 + 1e-7).any():
        raise ValueError(f"{name}: probabilities are outside [0, 1].")
    if not np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-5):
        raise ValueError(f"{name}: probabilities do not sum to one.")
    return out


corrected_path = find_one(["MITBIH_MLII_DualLead_Dataset_corrected.csv"])
beat_table_path = find_one([
    "validated_reconstructed_beat_table.csv.gz",
    "validated_reconstructed_beat_table.csv",
])

# Recreate the M0/M2 row ordering exactly: AAMI filter first, then Row_Index.
data = pd.read_csv(corrected_path, low_memory=False)
data.columns = [str(c).strip().strip('"').strip("'").strip() for c in data.columns]
if data.shape != (106870, 404):
    raise ValueError(f"Unexpected corrected dataset shape: {data.shape}")
if any(c not in data.columns for c in SIGNAL_COLS):
    raise ValueError("Corrected dataset is missing one or more waveform columns.")

data["Record_ID"] = pd.to_numeric(data["Record_ID"], errors="raise").astype(int)
data["Label"] = data["Label"].astype(str).str.strip()
data["AAMI_Label"] = data["Label"].map(AAMI_MAP)
aami_df = data[data["AAMI_Label"].isin(CLASS_ORDER)].copy().reset_index(drop=True)
aami_df["Person_ID"] = [
    "person_201_202" if rid in (201, 202) else f"person_{rid}"
    for rid in aami_df["Record_ID"].to_numpy()
]
aami_df["Row_Index"] = np.arange(len(aami_df), dtype=np.int64)

counts = aami_df["AAMI_Label"].value_counts().reindex(CLASS_ORDER, fill_value=0).to_dict()
assert len(aami_df) == EXPECTED_BEATS, len(aami_df)
assert aami_df["Record_ID"].nunique() == EXPECTED_RECORDS
assert aami_df["Person_ID"].nunique() == EXPECTED_PEOPLE
assert counts == EXPECTED_COUNTS, counts
assert np.isfinite(aami_df[SIGNAL_COLS].to_numpy(dtype=np.float32)).all()

beat_table = pd.read_csv(beat_table_path, compression="infer", low_memory=False)
beat_table.columns = [str(c).strip().strip('"').strip("'").strip() for c in beat_table.columns]
required_m1 = ["Record_ID", "Person_ID", "Sample_Index", "Original_Symbol", "AAMI_Label"] + SIGNAL_COLS
missing_m1 = [c for c in required_m1 if c not in beat_table.columns]
if missing_m1:
    raise ValueError(f"M1 beat table is missing columns: {missing_m1[:15]}")

beat_table["Record_ID"] = pd.to_numeric(beat_table["Record_ID"], errors="raise").astype(int)
beat_table["Sample_Index"] = pd.to_numeric(beat_table["Sample_Index"], errors="raise").astype(np.int64)
for col in ["Person_ID", "AAMI_Label", "Original_Symbol"]:
    beat_table[col] = beat_table[col].astype(str).str.strip()
if len(beat_table) != EXPECTED_BEATS:
    raise ValueError(f"M1 beat table has {len(beat_table)} rows, expected {EXPECTED_BEATS}.")
if beat_table[["Record_ID", "Sample_Index"]].duplicated().any():
    raise ValueError("Duplicate (Record_ID, Sample_Index) keys in M1 table.")
for col in ["Record_ID", "AAMI_Label", "Person_ID"]:
    if not np.array_equal(beat_table[col].to_numpy(), aami_df[col].to_numpy()):
        raise ValueError(f"M1 {col} does not match corrected dataset row-by-row. STOP.")

processed = aami_df[SIGNAL_COLS].to_numpy(dtype=np.float32)
reconstructed = beat_table[SIGNAL_COLS].to_numpy(dtype=np.float32)
if not np.array_equal(processed, reconstructed):
    mismatch = np.argwhere(processed != reconstructed)
    raise ValueError(f"M1 waveform mismatch. First mismatch: {mismatch[0].tolist()}")

aami_df["Sample_Index"] = beat_table["Sample_Index"].to_numpy(dtype=np.int64)
aami_df["Original_Symbol"] = beat_table["Original_Symbol"].to_numpy()

print("\nCORRECTED DATA AND M1 POSITIONS — PASS")
print("Valid beats:", len(aami_df), "| Records:", aami_df.Record_ID.nunique(), "| People:", aami_df.Person_ID.nunique())
print("Class counts:", counts)
print("All 402 waveform values match row-by-row: PASS")

FOUND: /kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-corrected-v1/MITBIH_MLII_DualLead_Dataset_corrected.csv
FOUND: /kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-corrected-v1/validated_reconstructed_beat_table.csv

CORRECTED DATA AND M1 POSITIONS — PASS
Valid beats: 105047 | Records: 46 | People: 45
Class counts: {'N': 90341, 'S': 2781, 'V': 7229, 'F': 802, 'Q': 3894}
All 402 waveform values match row-by-row: PASS


In [3]:
# Restore and verify the exact M0 person-grouped folds. Never regenerate splits.
row_path = find_one(["m0_row_fold_membership.csv.gz", "m0_row_fold_membership.csv"])
person_path = find_one(["m0_person_fold_membership.csv"])
fingerprint_path = find_one(["m0_split_fingerprint.txt"])

row_fold = pd.read_csv(row_path, compression="infer")
person_fold = pd.read_csv(person_path)
for col in ["Row_Index", "Repeat", "Fold"]:
    row_fold[col] = pd.to_numeric(row_fold[col], errors="raise").astype(int)
for col in ["Repeat", "Fold"]:
    person_fold[col] = pd.to_numeric(person_fold[col], errors="raise").astype(int)
person_fold["Person_ID"] = person_fold["Person_ID"].astype(str).str.strip()

computed_fp = hashlib.sha256(
    person_fold.sort_values(["Repeat", "Person_ID"]).to_csv(index=False, lineterminator="\n").encode("utf-8")
).hexdigest()
saved_fp = fingerprint_path.read_text().strip()
assert saved_fp == EXPECTED_SPLIT_FINGERPRINT
assert computed_fp == EXPECTED_SPLIT_FINGERPRINT
assert len(row_fold) == EXPECTED_BEATS * N_REPEATS
assert len(person_fold) == EXPECTED_PEOPLE * N_REPEATS
assert not row_fold.duplicated(["Repeat", "Row_Index"]).any()
assert not person_fold.duplicated(["Repeat", "Person_ID"]).any()
assert set(row_fold.Row_Index) == set(aami_df.Row_Index)
assert set(person_fold.Person_ID) == set(aami_df.Person_ID)

row_to_person = aami_df.set_index("Row_Index")["Person_ID"]
row_fold["Person_ID"] = row_fold["Row_Index"].map(row_to_person)
if row_fold["Person_ID"].isna().any():
    raise ValueError("Some Row_Index values have no Person_ID mapping.")
joined = row_fold.merge(
    person_fold, on=["Repeat", "Person_ID"], suffixes=("_row", "_person"), how="left", validate="many_to_one"
)
if joined["Fold_person"].isna().any() or not (joined["Fold_row"] == joined["Fold_person"]).all():
    raise ValueError("Row-level and person-level fold assignments disagree.")

row_to_position = {int(r): i for i, r in enumerate(aami_df.Row_Index.to_numpy())}
row_fold["Position"] = row_fold["Row_Index"].map(row_to_position).astype(int)
fold_assignment_by_repeat = {}
for repeat in range(1, N_REPEATS + 1):
    current = row_fold[row_fold["Repeat"] == repeat]
    assignment = np.full(EXPECTED_BEATS, -1, dtype=np.int8)
    assignment[current["Position"].to_numpy()] = current["Fold"].to_numpy()
    if not np.isin(assignment, [1, 2, 3, 4, 5]).all():
        raise ValueError(f"Incomplete fold assignment in repeat {repeat}.")
    if current.groupby("Person_ID")["Fold"].nunique().max() != 1:
        raise ValueError(f"Person-level leakage in repeat {repeat}.")
    record_check = current.merge(aami_df[["Row_Index", "Record_ID"]], on="Row_Index", validate="many_to_one")
    if record_check.groupby("Record_ID")["Fold"].nunique().max() != 1:
        raise ValueError(f"Record-level leakage in repeat {repeat}.")
    fold_assignment_by_repeat[repeat] = assignment

print("ORIGINAL M0 SPLITS RESTORED — PASS")
print("Split fingerprint:", computed_fp)
print("New splits generated: NO")

FOUND: /kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-corrected-v1/m0_row_fold_membership.csv
FOUND: /kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-corrected-v1/m0_person_fold_membership.csv
FOUND: /kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-corrected-v1/m0_split_fingerprint.txt
ORIGINAL M0 SPLITS RESTORED — PASS
Split fingerprint: 5a07f73452ca78af0b212db5e9f060fc4fcc66f49374c2462249189b847440d9
New splits generated: NO


In [4]:
# Load and validate the M0 reference and M2 Version A baseline OOF predictions.
reference_oof_path = find_one([
    "m0_corrected_oof_predictions_REFERENCE_MODEL.csv.gz",
    "m0_corrected_oof_predictions_REFERENCE_MODEL.csv",
])
m2_oof_path = find_one([
    "m2_oof_predictions_REFERENCE_MODEL_PLUS_RR_FEATURES.csv.gz",
    "m2_oof_predictions_REFERENCE_MODEL_PLUS_RR_FEATURES.csv",
])

reference_oof = validate_oof(
    pd.read_csv(reference_oof_path, compression="infer"), "M0 REFERENCE OOF", aami_df, row_fold
)
m2_oof = validate_oof(
    pd.read_csv(m2_oof_path, compression="infer"), "M2 VERSION A OOF", aami_df, row_fold
)

print("M0 REFERENCE OOF LOADED AND ALIGNED — PASS | Rows:", len(reference_oof))
print("M2 VERSION A OOF LOADED AND ALIGNED — PASS | Rows:", len(m2_oof))

FOUND: /kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-corrected-v1/m0_corrected_oof_predictions_REFERENCE_MODEL.csv/m0_corrected_oof_predictions_REFERENCE_MODEL.csv
FOUND: /kaggle/input/notebooks/aakashshah3799/05-m2-rr-features-revised-boundary-aware/ECG_Project/05_M2_RR_FEATURES/m2_oof_predictions_REFERENCE_MODEL_PLUS_RR_FEATURES.csv.gz
M0 REFERENCE OOF LOADED AND ALIGNED — PASS | Rows: 525235
M2 VERSION A OOF LOADED AND ALIGNED — PASS | Rows: 525235


In [5]:
# Load the selected M2 Version A RR table and align by Row_Index.
rr_path = find_one([
    "m2_rr_features_with_sample_positions.csv.gz",
    "m2_rr_features_with_sample_positions.csv",
])
rr_table = pd.read_csv(rr_path, compression="infer", low_memory=False)
rr_required = ["Row_Index", "Record_ID", "Person_ID", "Sample_Index", "AAMI_Label"] + RR_FEATURES
rr_missing_cols = [col for col in rr_required if col not in rr_table.columns]
if rr_missing_cols:
    raise ValueError(f"M2 RR table is missing columns: {rr_missing_cols}")
for col in ["Row_Index", "Record_ID", "Sample_Index"]:
    rr_table[col] = pd.to_numeric(rr_table[col], errors="raise").astype(np.int64)
for col in ["Person_ID", "AAMI_Label"]:
    rr_table[col] = rr_table[col].astype(str).str.strip()
if len(rr_table) != EXPECTED_BEATS or rr_table["Row_Index"].duplicated().any():
    raise ValueError("M2 RR table row count or Row_Index uniqueness is invalid.")
if set(rr_table["Row_Index"]) != set(aami_df["Row_Index"]):
    raise ValueError("M2 RR table Row_Index values do not match the M1 model rows.")
rr_aligned = rr_table.set_index("Row_Index").loc[aami_df["Row_Index"].to_numpy()]
for col in ["Record_ID", "Sample_Index", "Person_ID", "AAMI_Label"]:
    if not np.array_equal(rr_aligned[col].to_numpy(), aami_df[col].to_numpy()):
        raise ValueError(f"M2 RR table {col} does not align with source data. STOP.")

X_RR = rr_aligned[RR_FEATURES].to_numpy(dtype=np.float32)
if np.isinf(X_RR).any():
    raise ValueError("Infinite RR feature values found.")
observed_missing = dict(zip(RR_FEATURES, np.isnan(X_RR).sum(axis=0).astype(int).tolist()))
observed_rows_missing = int(np.isnan(X_RR).any(axis=1).sum())
if observed_missing != EXPECTED_RR_MISSING or observed_rows_missing != EXPECTED_ROWS_WITH_MISSING_RR:
    raise ValueError(
        "Attached RR table is not the verified M2 Version A table.\n"
        f"Observed missing by feature: {observed_missing}\nExpected: {EXPECTED_RR_MISSING}\n"
        f"Rows with any missing: {observed_rows_missing}; expected {EXPECTED_ROWS_WITH_MISSING_RR}."
    )
print("M2 VERSION A RR TABLE VERIFIED — PASS")
print("Missing per RR feature:", observed_missing)
print("Rows with any missing RR feature:", observed_rows_missing)

FOUND: /kaggle/input/notebooks/aakashshah3799/05-m2-rr-features-revised-boundary-aware/ECG_Project/05_M2_RR_FEATURES/m2_rr_features_with_sample_positions.csv
M2 VERSION A RR TABLE VERIFIED — PASS
Missing per RR feature: {'pre_RR': 716, 'post_RR': 719, 'mean_previous_10_RR': 259, 'pre_RR_div_mean_previous_10_RR': 946, 'post_RR_div_pre_RR': 1385}
Rows with any missing RR feature: 1614


In [6]:
# Recreate the exact 12 reference features and append the five M2 Version A RR features.
wave = aami_df[[f"MLII_{i}" for i in range(201)]].to_numpy(dtype=np.float32)
means = wave.mean(axis=1, keepdims=True)
stds = wave.std(axis=1, keepdims=True)
stds[stds == 0] = 1.0
normalized = (wave - means) / stds
d1 = np.diff(normalized, axis=1)
d2 = np.diff(normalized, n=2, axis=1)
q25, q50, q75 = np.percentile(normalized, [25, 50, 75], axis=1)
REFERENCE_FEATURE_NAMES = [
    "d1_mean", "d1_std", "d1_max", "d1_min", "d1_abs_mean", "d1_energy",
    "d2_std", "d2_abs_mean", "normalized_median", "normalized_IQR", "normalized_q25", "normalized_q75",
]
X_REFERENCE = np.column_stack([
    d1.mean(axis=1), d1.std(axis=1), d1.max(axis=1), d1.min(axis=1),
    np.abs(d1).mean(axis=1), np.square(d1).mean(axis=1),
    d2.std(axis=1), np.abs(d2).mean(axis=1), q50, q75 - q25, q25, q75,
]).astype(np.float32)
FEATURE_NAMES = REFERENCE_FEATURE_NAMES + RR_FEATURES
X = np.column_stack([X_REFERENCE, X_RR]).astype(np.float32)
y_text = aami_df["AAMI_Label"].to_numpy(dtype=str)
y = np.array([CLASS_TO_INT[label] for label in y_text], dtype=np.int32)
person_ids = aami_df["Person_ID"].to_numpy(dtype=str)
record_ids = aami_df["Record_ID"].to_numpy(dtype=np.int64)
row_ids = aami_df["Row_Index"].to_numpy(dtype=np.int64)

assert X_REFERENCE.shape == (EXPECTED_BEATS, 12)
assert X_RR.shape == (EXPECTED_BEATS, 5)
assert X.shape == (EXPECTED_BEATS, 17)
assert np.isfinite(X_REFERENCE).all()
assert not np.isinf(X).any()
assert np.array_equal(np.isnan(X[:, 12:]), np.isnan(X_RR))

feature_check = pd.DataFrame([
    {"Matrix": "REFERENCE", "Rows": len(y), "Features": X_REFERENCE.shape[1], "Missing_values": int(np.isnan(X_REFERENCE).sum())},
    {"Matrix": "M2 VERSION A BASELINE (REFERENCE + RR)", "Rows": len(y), "Features": X.shape[1], "Missing_values": int(np.isnan(X).sum())},
])
display(feature_check)
feature_check.to_csv(OUTPUT_DIR / "m4_xgb_feature_matrix_check.csv", index=False)

feature_manifest = {
    "phase": "M4 — XGBOOST CLASS WEIGHTING",
    "selected_feature_source": "M2 Version A",
    "feature_names": FEATURE_NAMES,
    "feature_count": len(FEATURE_NAMES),
    "valid_beats": EXPECTED_BEATS,
    "rr_table": str(rr_path),
    "m2_baseline_oof": str(m2_oof_path),
    "split_fingerprint": EXPECTED_SPLIT_FINGERPRINT,
    "splits_regenerated": False,
    "missing_rr_cells": int(np.isnan(X_RR).sum()),
    "xgboost_version": xgb.__version__,
    "xgboost_parameters": XGB_PARAMS,
    "num_boost_round": NUM_BOOST_ROUND,
    "balanced_weight_method": "n_train / (n_present_classes * count_in_training_fold_class)",
}
(OUTPUT_DIR / "m4_xgb_feature_manifest.json").write_text(json.dumps(feature_manifest, indent=2))
print("FEATURE MATRIX CONSTRUCTION COMPLETE — PASS")
print("Rows:", len(y), "| Features:", X.shape[1], "| RR missing cells:", int(np.isnan(X_RR).sum()))
print("Training switch:", RUN_MODEL_TRAINING)

,Matrix,Rows,Features,Missing_values
0,REFERENCE,105047,12,0
1,M2 VERSION A BASELINE (REFERENCE + RR),105047,17,4025


FEATURE MATRIX CONSTRUCTION COMPLETE — PASS
Rows: 105047 | Features: 17 | RR missing cells: 4025
Training switch: True


## XGBoost smoke test

This small synthetic check verifies that the installed XGBoost build accepts missing feature values and training-row weights before the full cross-validation run. It is not a model-performance experiment.

In [7]:
_rng = np.random.default_rng(2026)
_smoke_X = _rng.normal(size=(120, X.shape[1])).astype(np.float32)
_smoke_X[0, 12] = np.nan
_smoke_y = np.tile(np.arange(len(CLASS_ORDER), dtype=np.int32), 24)
_smoke_counts = np.bincount(_smoke_y, minlength=len(CLASS_ORDER))
_smoke_weights = len(_smoke_y) / (len(CLASS_ORDER) * _smoke_counts[_smoke_y])
_smoke_dtrain = xgb.DMatrix(_smoke_X, label=_smoke_y, weight=_smoke_weights, feature_names=FEATURE_NAMES)
_smoke_model = xgb.train(XGB_PARAMS, _smoke_dtrain, num_boost_round=2, verbose_eval=False)
_smoke_probs = _smoke_model.predict(xgb.DMatrix(_smoke_X[:10], feature_names=FEATURE_NAMES))
assert _smoke_probs.shape == (10, len(CLASS_ORDER))
assert np.isfinite(_smoke_probs).all()
assert np.allclose(_smoke_probs.sum(axis=1), 1.0, atol=1e-5)
print("XGBOOST MULTICLASS + NaN + SAMPLE-WEIGHT SMOKE TEST — PASS")

XGBOOST MULTICLASS + NaN + SAMPLE-WEIGHT SMOKE TEST — PASS


## Train both XGBoost configurations

`XGB_CLASS_WEIGHT_BALANCED_SAMPLE_WEIGHT` uses balanced weights computed from **only the current training fold**. The unweighted configuration passes no weights. Both use the same CPU histogram-based tree settings.

In [8]:
if RUN_MODEL_TRAINING:
    def make_balanced_sample_weights(y_train_encoded):
        counts_train = np.bincount(y_train_encoded, minlength=len(CLASS_ORDER))
        present = counts_train > 0
        n_present = int(present.sum())
        if n_present < 2:
            raise ValueError("Fewer than two classes are present in a training fold.")
        # Equivalent to balanced class weights over the classes present in this training fold.
        weights_by_class = np.zeros(len(CLASS_ORDER), dtype=np.float64)
        weights_by_class[present] = len(y_train_encoded) / (n_present * counts_train[present])
        weights = weights_by_class[y_train_encoded]
        if not np.isfinite(weights).all() or (weights <= 0).any():
            raise ValueError("Invalid balanced sample weights.")
        return weights, counts_train, weights_by_class

    def train_experiment(name, use_balanced_weights):
        pred_path = OUTPUT_DIR / f"m4_xgb_oof_predictions_{name}.csv.gz"
        audit_path = OUTPUT_DIR / f"m4_xgb_fold_run_audit_{name}.csv"
        prediction_parts = []
        audit_rows = []
        start = time.time()

        for repeat in range(1, N_REPEATS + 1):
            assignment = fold_assignment_by_repeat[repeat]
            for fold in range(1, N_SPLITS + 1):
                val_idx = np.flatnonzero(assignment == fold)
                train_idx = np.flatnonzero(assignment != fold)
                if not set(person_ids[train_idx]).isdisjoint(set(person_ids[val_idx])):
                    raise ValueError("Person leakage detected during XGBoost training.")
                if not set(record_ids[train_idx]).isdisjoint(set(record_ids[val_idx])):
                    raise ValueError("Record leakage detected during XGBoost training.")

                y_train = y[train_idx]
                if use_balanced_weights:
                    train_weights, train_class_counts, weights_by_class = make_balanced_sample_weights(y_train)
                else:
                    train_weights = None
                    train_class_counts = np.bincount(y_train, minlength=len(CLASS_ORDER))
                    weights_by_class = np.ones(len(CLASS_ORDER), dtype=np.float64)

                dtrain = xgb.DMatrix(
                    X[train_idx],
                    label=y_train,
                    weight=train_weights,
                    feature_names=FEATURE_NAMES,
                )
                dvalid = xgb.DMatrix(X[val_idx], feature_names=FEATURE_NAMES)
                booster = xgb.train(
                    params=XGB_PARAMS,
                    dtrain=dtrain,
                    num_boost_round=NUM_BOOST_ROUND,
                    verbose_eval=False,
                )
                probabilities = np.asarray(booster.predict(dvalid), dtype=np.float32)
                if probabilities.shape != (len(val_idx), len(CLASS_ORDER)):
                    raise ValueError(f"Unexpected XGBoost probability shape: {probabilities.shape}")
                if not np.isfinite(probabilities).all() or not np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-5):
                    raise ValueError("XGBoost returned invalid multiclass probabilities.")
                predicted_codes = probabilities.argmax(axis=1)
                predicted_labels = np.array([INT_TO_CLASS[int(code)] for code in predicted_codes], dtype=object)

                part = pd.DataFrame({
                    "Experiment": name,
                    "Repeat": repeat,
                    "Fold": fold,
                    "Row_Index": row_ids[val_idx],
                    "Record_ID": record_ids[val_idx],
                    "Person_ID": person_ids[val_idx],
                    "True_Label": y_text[val_idx],
                    "Predicted_Label": predicted_labels,
                })
                for j, cls in enumerate(CLASS_ORDER):
                    part[f"Prob_{cls}"] = probabilities[:, j]
                prediction_parts.append(part)

                audit = {
                    "Experiment": name,
                    "Repeat": repeat,
                    "Fold": fold,
                    "Training_Beats": len(train_idx),
                    "Validation_Beats": len(val_idx),
                    "Training_People": len(set(person_ids[train_idx])),
                    "Validation_People": len(set(person_ids[val_idx])),
                    "Training_Records": len(set(record_ids[train_idx])),
                    "Validation_Records": len(set(record_ids[val_idx])),
                    "Weighting": "balanced_sample_weight" if use_balanced_weights else "none",
                    "Training_Weight_Min": float(train_weights.min()) if train_weights is not None else 1.0,
                    "Training_Weight_Max": float(train_weights.max()) if train_weights is not None else 1.0,
                    "Training_Weight_Mean": float(train_weights.mean()) if train_weights is not None else 1.0,
                    "Absent_Training_Classes": ";".join(
                        CLASS_ORDER[i] for i, count in enumerate(train_class_counts) if count == 0
                    ),
                    "Training_Class_Counts": json.dumps({
                        CLASS_ORDER[i]: int(count) for i, count in enumerate(train_class_counts)
                    }),
                    "Training_Class_Weights": json.dumps({
                        CLASS_ORDER[i]: float(weights_by_class[i]) for i in range(len(CLASS_ORDER))
                    }),
                    "XGBoost_Version": xgb.__version__,
                    "Num_Boost_Round": NUM_BOOST_ROUND,
                }
                audit_rows.append(audit)
                print(
                    f"Completed {name}: Repeat {repeat}/{N_REPEATS}, Fold {fold}/{N_SPLITS} "
                    f"({time.time() - start:.0f}s elapsed)",
                    flush=True,
                )

        pred_df = pd.concat(prediction_parts, ignore_index=True)
        audit_df = pd.DataFrame(audit_rows)
        if len(pred_df) != EXPECTED_BEATS * N_REPEATS:
            raise ValueError(f"{name}: incomplete OOF predictions: {len(pred_df)} rows.")
        if len(audit_df) != N_REPEATS * N_SPLITS:
            raise ValueError(f"{name}: expected 25 fold audit rows, found {len(audit_df)}.")
        if pred_df.duplicated(["Repeat", "Row_Index"]).any():
            raise ValueError(f"{name}: duplicate OOF (Repeat, Row_Index) pairs.")
        if not pred_df.groupby("Repeat")["Row_Index"].nunique().eq(EXPECTED_BEATS).all():
            raise ValueError(f"{name}: one or more repeats are missing OOF rows.")
        if not np.allclose(pred_df[PROB_COLS].sum(axis=1), 1.0, atol=1e-5):
            raise ValueError(f"{name}: saved probability rows do not sum to one.")

        pred_df.to_csv(pred_path, index=False, compression="gzip")
        audit_df.to_csv(audit_path, index=False)
        return pred_df, audit_df

    xgb_predictions = {}
    xgb_fold_audits = {}
    for experiment_name, weighted in [
        (UNWEIGHTED_NAME, False),
        (WEIGHTED_NAME, True),
    ]:
        xgb_predictions[experiment_name], xgb_fold_audits[experiment_name] = train_experiment(
            experiment_name, weighted
        )
    print("\nBOTH XGBOOST TRAINING CONFIGURATIONS COMPLETE.")
else:
    print("Training skipped. Set RUN_MODEL_TRAINING=True to run both XGBoost configurations.")

Completed XGB_CLASS_WEIGHT_NONE: Repeat 1/5, Fold 1/5 (7s elapsed)
Completed XGB_CLASS_WEIGHT_NONE: Repeat 1/5, Fold 2/5 (15s elapsed)
Completed XGB_CLASS_WEIGHT_NONE: Repeat 1/5, Fold 3/5 (22s elapsed)
Completed XGB_CLASS_WEIGHT_NONE: Repeat 1/5, Fold 4/5 (29s elapsed)
Completed XGB_CLASS_WEIGHT_NONE: Repeat 1/5, Fold 5/5 (36s elapsed)
Completed XGB_CLASS_WEIGHT_NONE: Repeat 2/5, Fold 1/5 (44s elapsed)
Completed XGB_CLASS_WEIGHT_NONE: Repeat 2/5, Fold 2/5 (52s elapsed)
Completed XGB_CLASS_WEIGHT_NONE: Repeat 2/5, Fold 3/5 (59s elapsed)
Completed XGB_CLASS_WEIGHT_NONE: Repeat 2/5, Fold 4/5 (66s elapsed)
Completed XGB_CLASS_WEIGHT_NONE: Repeat 2/5, Fold 5/5 (74s elapsed)
Completed XGB_CLASS_WEIGHT_NONE: Repeat 3/5, Fold 1/5 (82s elapsed)
Completed XGB_CLASS_WEIGHT_NONE: Repeat 3/5, Fold 2/5 (89s elapsed)
Completed XGB_CLASS_WEIGHT_NONE: Repeat 3/5, Fold 3/5 (96s elapsed)
Completed XGB_CLASS_WEIGHT_NONE: Repeat 3/5, Fold 4/5 (104s elapsed)
Completed XGB_CLASS_WEIGHT_NONE: Repeat 3/5, Fol

In [9]:
if RUN_MODEL_TRAINING:
    def evaluate_predictions(pred, name):
        metric_rows, class_rows, confusion_rows, s_rows = [], [], [], []
        for repeat in range(1, N_REPEATS + 1):
            part = pred[pred["Repeat"] == repeat]
            yt = part["True_Label"].to_numpy()
            yp = part["Predicted_Label"].to_numpy()
            precision, recall, f1, support = precision_recall_fscore_support(
                yt, yp, labels=CLASS_ORDER, zero_division=0
            )
            four = part[part["True_Label"].isin(FOUR_CLASS_ORDER)]
            metric_rows.append({
                "Experiment": name,
                "Repeat": repeat,
                "Macro_F1_5_Class": f1_score(yt, yp, labels=CLASS_ORDER, average="macro", zero_division=0),
                "Macro_F1_4_Class_N_S_V_Q": f1_score(
                    four["True_Label"], four["Predicted_Label"], labels=FOUR_CLASS_ORDER,
                    average="macro", zero_division=0,
                ),
                "Balanced_Accuracy": balanced_accuracy_score(yt, yp),
                "Accuracy": accuracy_score(yt, yp),
            })
            class_rows.append(pd.DataFrame({
                "Experiment": name, "Repeat": repeat, "Class": CLASS_ORDER,
                "Precision": precision, "Recall": recall, "F1": f1, "Support": support,
            }))
            cm = confusion_matrix(yt, yp, labels=CLASS_ORDER)
            cm_df = pd.DataFrame(
                cm,
                index=[f"Actual_{c}" for c in CLASS_ORDER],
                columns=[f"Predicted_{c}" for c in CLASS_ORDER],
            )
            cm_df.insert(0, "Actual_Class", cm_df.index)
            cm_df["Repeat"] = repeat
            cm_df["Experiment"] = name
            confusion_rows.append(cm_df.reset_index(drop=True))
            mean_s_on_s = part.loc[part["True_Label"] == "S", "Prob_S"].mean()
            mean_s_on_n = part.loc[part["True_Label"] == "N", "Prob_S"].mean()
            s_rows.append({
                "Experiment": name, "Repeat": repeat,
                "Mean_Prob_S_on_true_S": mean_s_on_s,
                "Mean_Prob_S_on_true_N": mean_s_on_n,
                "Difference_S_minus_N": mean_s_on_s - mean_s_on_n,
            })

        repeat_metrics = pd.DataFrame(metric_rows)
        per_class = pd.concat(class_rows, ignore_index=True)
        confusions = pd.concat(confusion_rows, ignore_index=True)
        s_diagnostic = pd.DataFrame(s_rows)
        headline = {
            "Experiment": name,
            "Headline_Macro_F1_5_Class": repeat_metrics["Macro_F1_5_Class"].mean(),
            "Macro_F1_4_Class_N_S_V_Q": repeat_metrics["Macro_F1_4_Class_N_S_V_Q"].mean(),
            "Balanced_Accuracy": repeat_metrics["Balanced_Accuracy"].mean(),
            "Accuracy": repeat_metrics["Accuracy"].mean(),
        }
        return headline, repeat_metrics, per_class, confusions, s_diagnostic

    predictions_for_evaluation = {
        "REFERENCE_MODEL": reference_oof,
        BASELINE_NAME: m2_oof,
        UNWEIGHTED_NAME: xgb_predictions[UNWEIGHTED_NAME],
        WEIGHTED_NAME: xgb_predictions[WEIGHTED_NAME],
    }
    evaluations = {}
    headline_rows = []
    for name, pred in predictions_for_evaluation.items():
        evaluation = evaluate_predictions(pred, name)
        evaluations[name] = evaluation
        headline_rows.append(evaluation[0])
        print("\n", name, "— metrics by repeat")
        display(evaluation[1])
        print("Per-class metrics")
        display(evaluation[2])
        print("S probability diagnostic")
        display(evaluation[4])
        evaluation[1].to_csv(OUTPUT_DIR / f"m4_xgb_repeat_metrics_{name}.csv", index=False)
        evaluation[2].to_csv(OUTPUT_DIR / f"m4_xgb_per_class_metrics_{name}.csv", index=False)
        evaluation[3].to_csv(OUTPUT_DIR / f"m4_xgb_confusion_matrices_{name}.csv", index=False)
        evaluation[4].to_csv(OUTPUT_DIR / f"m4_xgb_s_probability_diagnostic_{name}.csv", index=False)

    m4_xgb_headline = pd.DataFrame(headline_rows)
    print("\nM4 XGBOOST HEADLINE RESULTS")
    display(m4_xgb_headline)
    m4_xgb_headline.to_csv(OUTPUT_DIR / "m4_xgb_headline_results.csv", index=False)

    per_class_all = pd.concat([ev[2] for ev in evaluations.values()], ignore_index=True)
    per_class_mean = (
        per_class_all.groupby(["Experiment", "Class"], sort=False)[["Precision", "Recall", "F1"]]
        .mean().reset_index()
    )
    print("\nPER-CLASS RESULTS, MEAN OVER REPEATS")
    display(per_class_mean)
    per_class_mean.to_csv(OUTPUT_DIR / "m4_xgb_per_class_mean_over_repeats.csv", index=False)
else:
    print("Metrics are deferred until both XGBoost configurations have trained.")


 REFERENCE_MODEL — metrics by repeat


,Experiment,Repeat,Macro_F1_5_Class,Macro_F1_4_Class_N_S_V_Q,Balanced_Accuracy,Accuracy
0,REFERENCE_MODEL,1,0.518445,0.651288,0.535339,0.913344
1,REFERENCE_MODEL,2,0.316230,0.397515,0.364539,0.889211
2,REFERENCE_MODEL,3,0.520605,0.653948,0.540737,0.912877
3,REFERENCE_MODEL,4,0.312341,0.392648,0.362271,0.875903
4,REFERENCE_MODEL,5,0.508729,0.638717,0.530906,0.901501


Per-class metrics


,Experiment,Repeat,Class,Precision,Recall,F1,Support
0,REFERENCE_MODEL,1,N,0.948918,0.956365,0.952627,90341
1,REFERENCE_MODEL,1,S,0.015385,0.000360,0.000703,2781
2,REFERENCE_MODEL,1,V,0.610891,0.853507,0.712101,7229
3,REFERENCE_MODEL,1,F,0.000000,0.000000,0.000000,802
4,REFERENCE_MODEL,1,Q,0.996162,0.866461,0.926796,3894
5,REFERENCE_MODEL,2,N,0.949499,0.965453,0.957409,90341
6,REFERENCE_MODEL,2,S,0.000000,0.000000,0.000000,2781
7,REFERENCE_MODEL,2,V,0.488591,0.855997,0.622097,7229
8,REFERENCE_MODEL,2,F,0.002404,0.001247,0.001642,802
9,REFERENCE_MODEL,2,Q,0.000000,0.000000,0.000000,3894


S probability diagnostic


,Experiment,Repeat,Mean_Prob_S_on_true_S,Mean_Prob_S_on_true_N,Difference_S_minus_N
0,REFERENCE_MODEL,1,0.046721,0.043845,0.002876
1,REFERENCE_MODEL,2,0.034694,0.046034,-0.011340
2,REFERENCE_MODEL,3,0.042271,0.043655,-0.001385
3,REFERENCE_MODEL,4,0.042913,0.056732,-0.013819
4,REFERENCE_MODEL,5,0.040309,0.045288,-0.004979


... (18 outputs left out)


M4 XGBOOST HEADLINE RESULTS


,Experiment,Headline_Macro_F1_5_Class,Macro_F1_4_Class_N_S_V_Q,Balanced_Accuracy,Accuracy
0,REFERENCE_MODEL,0.435270,0.546823,0.466758,0.898567
1,M2_VERSION_A_REFERENCE_PLUS_RR,0.501716,0.625019,0.504764,0.910155
2,XGB_CLASS_WEIGHT_NONE,0.504282,0.628236,0.513442,0.905048
3,XGB_CLASS_WEIGHT_BALANCED_SAMPLE_WEIGHT,0.501482,0.623247,0.532226,0.879351



PER-CLASS RESULTS, MEAN OVER REPEATS


,Experiment,Class,Precision,Recall,F1
0,REFERENCE_MODEL,N,0.947087,0.954041,0.950541
1,REFERENCE_MODEL,S,0.005955,0.000216,0.000415
2,REFERENCE_MODEL,V,0.547300,0.849357,0.663805
3,REFERENCE_MODEL,F,0.001069,0.000748,0.000868
4,REFERENCE_MODEL,Q,0.596060,0.529430,0.560723
5,M2_VERSION_A_REFERENCE_PLUS_RR,N,0.943904,0.960226,0.951911
6,M2_VERSION_A_REFERENCE_PLUS_RR,S,0.360003,0.136929,0.197582
7,M2_VERSION_A_REFERENCE_PLUS_RR,V,0.717874,0.906875,0.801241
8,M2_VERSION_A_REFERENCE_PLUS_RR,F,0.031507,0.032170,0.021710
9,M2_VERSION_A_REFERENCE_PLUS_RR,Q,0.596220,0.487622,0.536134


## Record-level confidence intervals and paired retention tests

Paired comparisons use the same sampled `Record_ID`s for both models, preserving the paired design. The primary retention metric is 5-class Macro-F1.

In [10]:
if RUN_MODEL_TRAINING:
    METRIC_CLASSES = {
        "Macro_F1_5_Class": CLASS_ORDER,
        "Macro_F1_4_Class_N_S_V_Q": FOUR_CLASS_ORDER,
    }
    CLASS_INDEX = {c: i for i, c in enumerate(CLASS_ORDER)}

    def macro_f1_from_counts(matrix, scored_classes):
        ids = [CLASS_INDEX[c] for c in scored_classes]
        scored_rows = matrix[ids, :]
        f1_values = []
        for class_id in ids:
            tp = matrix[class_id, class_id]
            fn = matrix[class_id, :].sum() - tp
            fp = scored_rows[:, class_id].sum() - tp
            denominator = 2 * tp + fp + fn
            f1_values.append(0.0 if denominator == 0 else 2.0 * tp / denominator)
        return float(np.mean(f1_values))

    def build_blocks(pred):
        true_codes = pred["True_Label"].map(CLASS_INDEX).to_numpy()
        predicted_codes = pred["Predicted_Label"].map(CLASS_INDEX).to_numpy()
        if pd.isna(true_codes).any() or pd.isna(predicted_codes).any():
            raise ValueError("Unknown label found while preparing bootstrap blocks.")
        flat = true_codes.astype(np.int64) * len(CLASS_ORDER) + predicted_codes.astype(np.int64)
        blocks = {}
        repeats = pred["Repeat"].to_numpy()
        records = pred["Record_ID"].to_numpy()
        for repeat in np.unique(repeats):
            repeat_mask = repeats == repeat
            for record in np.unique(records[repeat_mask]):
                selection = repeat_mask & (records == record)
                blocks[(int(repeat), int(record))] = np.bincount(
                    flat[selection], minlength=len(CLASS_ORDER) ** 2
                ).reshape(len(CLASS_ORDER), len(CLASS_ORDER)).astype(np.int64)
        return blocks

    def point_score(blocks, metric):
        repeats = sorted({key[0] for key in blocks})
        scores = []
        for repeat in repeats:
            matrix = sum(value for (r, _), value in blocks.items() if r == repeat)
            scores.append(macro_f1_from_counts(matrix, METRIC_CLASSES[metric]))
        return float(np.mean(scores))

    def bootstrap_scores(blocks_list, metric, records, repeats):
        rng = np.random.default_rng(BOOTSTRAP_SEED)
        outputs = [[] for _ in blocks_list]
        for _ in range(BOOTSTRAP_SAMPLES):
            sampled_records = rng.choice(records, size=len(records), replace=True)
            for model_index, blocks in enumerate(blocks_list):
                repeat_scores = []
                for repeat in repeats:
                    matrix = sum(blocks[(int(repeat), int(record))] for record in sampled_records)
                    repeat_scores.append(macro_f1_from_counts(matrix, METRIC_CLASSES[metric]))
                outputs[model_index].append(float(np.mean(repeat_scores)))
        return [np.asarray(values) for values in outputs]

    def record_ci(pred, metric, name):
        blocks = build_blocks(pred)
        records = np.array(sorted(pred["Record_ID"].unique()))
        repeats = sorted(pred["Repeat"].unique())
        (scores,) = bootstrap_scores([blocks], metric, records, repeats)
        lower, upper = np.quantile(scores, [0.025, 0.975])
        return {
            "Experiment": name, "Metric": metric,
            "Point_Estimate": point_score(blocks, metric),
            "CI_95_Lower": float(lower), "CI_95_Upper": float(upper),
            "Bootstrap_Samples": BOOTSTRAP_SAMPLES,
            "Resampling_Unit": "Record_ID", "Bootstrap_Seed": BOOTSTRAP_SEED,
        }

    def paired_record_bootstrap(candidate, comparison, metric, candidate_name, comparison_name, is_retention_test):
        keys = ["Repeat", "Row_Index"]
        metadata = ["Record_ID", "Person_ID", "True_Label"]
        merged = candidate.merge(
            comparison[keys + metadata + ["Predicted_Label"]],
            on=keys, how="inner", suffixes=("_candidate", "_comparison"), validate="one_to_one",
        )
        if len(merged) != EXPECTED_BEATS * N_REPEATS:
            raise ValueError(f"{candidate_name} and {comparison_name} OOF predictions are not fully aligned.")
        for col in metadata:
            if not (
                merged[f"{col}_candidate"].astype(str) == merged[f"{col}_comparison"].astype(str)
            ).all():
                raise ValueError(f"Paired predictions disagree on {col}.")

        candidate_df = pd.DataFrame({
            "Repeat": merged["Repeat"],
            "Record_ID": merged["Record_ID_candidate"],
            "True_Label": merged["True_Label_candidate"],
            "Predicted_Label": merged["Predicted_Label_candidate"],
        })
        comparison_df = candidate_df.copy()
        comparison_df["Predicted_Label"] = merged["Predicted_Label_comparison"].to_numpy()
        candidate_blocks = build_blocks(candidate_df)
        comparison_blocks = build_blocks(comparison_df)
        records = np.array(sorted(candidate_df["Record_ID"].unique()))
        repeats = sorted(candidate_df["Repeat"].unique())
        difference = point_score(candidate_blocks, metric) - point_score(comparison_blocks, metric)
        candidate_scores, comparison_scores = bootstrap_scores(
            [candidate_blocks, comparison_blocks], metric, records, repeats
        )
        lower, upper = np.quantile(candidate_scores - comparison_scores, [0.025, 0.975])

        if is_retention_test:
            decision = "KEEP" if lower > 0 else f"NO MEASURABLE IMPROVEMENT — retain {comparison_name}"
            rule = "KEEP only if paired CI lower bound > 0"
        else:
            decision = "CONTEXT"
            rule = "Context only (not a primary retention test)"

        return {
            "Candidate": candidate_name,
            "Comparison": comparison_name,
            "Metric": metric,
            "Difference_Candidate_minus_Comparison": float(difference),
            "Paired_CI_95_Lower": float(lower),
            "Paired_CI_95_Upper": float(upper),
            "Bootstrap_Samples": BOOTSTRAP_SAMPLES,
            "Resampling_Unit": "Record_ID",
            "Bootstrap_Seed": BOOTSTRAP_SEED,
            "Retention_Rule": rule,
            "Decision": decision,
        }

    all_predictions = {
        "REFERENCE_MODEL": reference_oof,
        BASELINE_NAME: m2_oof,
        UNWEIGHTED_NAME: xgb_predictions[UNWEIGHTED_NAME],
        WEIGHTED_NAME: xgb_predictions[WEIGHTED_NAME],
    }

    ci_rows = []
    for name, pred in all_predictions.items():
        for metric in METRIC_CLASSES:
            ci_rows.append(record_ci(pred, metric, name))
    m4_xgb_ci = pd.DataFrame(ci_rows)

    comparisons = [
        (UNWEIGHTED_NAME, BASELINE_NAME, True),
        (WEIGHTED_NAME, BASELINE_NAME, True),
        (WEIGHTED_NAME, UNWEIGHTED_NAME, False),
        (UNWEIGHTED_NAME, "REFERENCE_MODEL", False),
        (WEIGHTED_NAME, "REFERENCE_MODEL", False),
    ]
    paired_rows = []
    for candidate_name, comparison_name, is_retention_test in comparisons:
        for metric in METRIC_CLASSES:
            paired_rows.append(paired_record_bootstrap(
                all_predictions[candidate_name], all_predictions[comparison_name],
                metric, candidate_name, comparison_name, is_retention_test,
            ))
    m4_xgb_paired = pd.DataFrame(paired_rows)

    print("M4 XGBOOST RECORD-LEVEL CONFIDENCE INTERVALS")
    display(m4_xgb_ci)
    print("\nM4 XGBOOST PAIRED COMPARISONS")
    display(m4_xgb_paired)

    primary_decisions = m4_xgb_paired[
        (m4_xgb_paired["Comparison"] == BASELINE_NAME)
        & (m4_xgb_paired["Metric"] == "Macro_F1_5_Class")
    ].copy()
    print("\nM4 XGBOOST PRIMARY DECISIONS — 5-CLASS MACRO-F1 VS M2 VERSION A")
    display(primary_decisions)

    m4_xgb_ci.to_csv(OUTPUT_DIR / "m4_xgb_record_bootstrap_ci.csv", index=False)
    m4_xgb_paired.to_csv(OUTPUT_DIR / "m4_xgb_paired_record_bootstrap.csv", index=False)
    primary_decisions.to_csv(OUTPUT_DIR / "m4_xgb_primary_decisions.csv", index=False)
else:
    print("Bootstrap confidence intervals and retention decisions are deferred until training is enabled.")

M4 XGBOOST RECORD-LEVEL CONFIDENCE INTERVALS


,Experiment,Metric,Point_Estimate,CI_95_Lower,CI_95_Upper,Bootstrap_Samples,Resampling_Unit,Bootstrap_Seed
0,REFERENCE_MODEL,Macro_F1_5_Class,0.435270,0.311816,0.467416,1000,Record_ID,20261105
1,REFERENCE_MODEL,Macro_F1_4_Class_N_S_V_Q,0.546823,0.393360,0.588774,1000,Record_ID,20261105
2,M2_VERSION_A_REFERENCE_PLUS_RR,Macro_F1_5_Class,0.501716,0.376369,0.557963,1000,Record_ID,20261105
3,M2_VERSION_A_REFERENCE_PLUS_RR,Macro_F1_4_Class_N_S_V_Q,0.625019,0.467384,0.697156,1000,Record_ID,20261105
4,XGB_CLASS_WEIGHT_NONE,Macro_F1_5_Class,0.504282,0.375312,0.560730,1000,Record_ID,20261105
5,XGB_CLASS_WEIGHT_NONE,Macro_F1_4_Class_N_S_V_Q,0.628236,0.467876,0.700547,1000,Record_ID,20261105
6,XGB_CLASS_WEIGHT_BALANCED_SAMPLE_WEIGHT,Macro_F1_5_Class,0.501482,0.375876,0.551331,1000,Record_ID,20261105
7,XGB_CLASS_WEIGHT_BALANCED_SAMPLE_WEIGHT,Macro_F1_4_Class_N_S_V_Q,0.623247,0.464017,0.687943,1000,Record_ID,20261105



M4 XGBOOST PAIRED COMPARISONS


,Candidate,Comparison,Metric,Difference_Candidate_minus_Comparison,Paired_CI_95_Lower,Paired_CI_95_Upper,Bootstrap_Samples,Resampling_Unit,Bootstrap_Seed,Retention_Rule,Decision
0,XGB_CLASS_WEIGHT_NONE,M2_VERSION_A_REFERENCE_PLUS_RR,Macro_F1_5_Class,0.002567,-0.004180,0.009493,1000,Record_ID,20261105,KEEP only if paired CI lower bound > 0,NO MEASURABLE IMPROVEMENT — retain M2_VERSION_...
1,XGB_CLASS_WEIGHT_NONE,M2_VERSION_A_REFERENCE_PLUS_RR,Macro_F1_4_Class_N_S_V_Q,0.003216,-0.003969,0.013006,1000,Record_ID,20261105,KEEP only if paired CI lower bound > 0,NO MEASURABLE IMPROVEMENT — retain M2_VERSION_...
2,XGB_CLASS_WEIGHT_BALANCED_SAMPLE_WEIGHT,M2_VERSION_A_REFERENCE_PLUS_RR,Macro_F1_5_Class,-0.000234,-0.017254,0.012564,1000,Record_ID,20261105,KEEP only if paired CI lower bound > 0,NO MEASURABLE IMPROVEMENT — retain M2_VERSION_...
3,XGB_CLASS_WEIGHT_BALANCED_SAMPLE_WEIGHT,M2_VERSION_A_REFERENCE_PLUS_RR,Macro_F1_4_Class_N_S_V_Q,-0.001772,-0.020681,0.012846,1000,Record_ID,20261105,KEEP only if paired CI lower bound > 0,NO MEASURABLE IMPROVEMENT — retain M2_VERSION_...
4,XGB_CLASS_WEIGHT_BALANCED_SAMPLE_WEIGHT,XGB_CLASS_WEIGHT_NONE,Macro_F1_5_Class,-0.002800,-0.018752,0.009701,1000,Record_ID,20261105,Context only (not a primary retention test),CONTEXT
5,XGB_CLASS_WEIGHT_BALANCED_SAMPLE_WEIGHT,XGB_CLASS_WEIGHT_NONE,Macro_F1_4_Class_N_S_V_Q,-0.004989,-0.023589,0.009170,1000,Record_ID,20261105,Context only (not a primary retention test),CONTEXT
6,XGB_CLASS_WEIGHT_NONE,REFERENCE_MODEL,Macro_F1_5_Class,0.069012,0.032638,0.118490,1000,Record_ID,20261105,Context only (not a primary retention test),CONTEXT
7,XGB_CLASS_WEIGHT_NONE,REFERENCE_MODEL,Macro_F1_4_Class_N_S_V_Q,0.081412,0.036577,0.143757,1000,Record_ID,20261105,Context only (not a primary retention test),CONTEXT
8,XGB_CLASS_WEIGHT_BALANCED_SAMPLE_WEIGHT,REFERENCE_MODEL,Macro_F1_5_Class,0.066212,0.028701,0.113453,1000,Record_ID,20261105,Context only (not a primary retention test),CONTEXT
9,XGB_CLASS_WEIGHT_BALANCED_SAMPLE_WEIGHT,REFERENCE_MODEL,Macro_F1_4_Class_N_S_V_Q,0.076424,0.031141,0.134830,1000,Record_ID,20261105,Context only (not a primary retention test),CONTEXT



M4 XGBOOST PRIMARY DECISIONS — 5-CLASS MACRO-F1 VS M2 VERSION A


,Candidate,Comparison,Metric,Difference_Candidate_minus_Comparison,Paired_CI_95_Lower,Paired_CI_95_Upper,Bootstrap_Samples,Resampling_Unit,Bootstrap_Seed,Retention_Rule,Decision
0,XGB_CLASS_WEIGHT_NONE,M2_VERSION_A_REFERENCE_PLUS_RR,Macro_F1_5_Class,0.002567,-0.004180,0.009493,1000,Record_ID,20261105,KEEP only if paired CI lower bound > 0,NO MEASURABLE IMPROVEMENT — retain M2_VERSION_...
2,XGB_CLASS_WEIGHT_BALANCED_SAMPLE_WEIGHT,M2_VERSION_A_REFERENCE_PLUS_RR,Macro_F1_5_Class,-0.000234,-0.017254,0.012564,1000,Record_ID,20261105,KEEP only if paired CI lower bound > 0,NO MEASURABLE IMPROVEMENT — retain M2_VERSION_...


In [11]:
if RUN_MODEL_TRAINING:
    checks = {
        "Corrected_dataset_selected": corrected_path.name == "MITBIH_MLII_DualLead_Dataset_corrected.csv",
        "M1_positions_loaded": bool(aami_df["Sample_Index"].notna().all()),
        "M1_waveforms_match_corrected_dataset": bool(np.array_equal(processed, reconstructed)),
        "105047_valid_beats": len(aami_df) == EXPECTED_BEATS,
        "Expected_class_counts": counts == EXPECTED_COUNTS,
        "46_records_45_people": aami_df.Record_ID.nunique() == EXPECTED_RECORDS and aami_df.Person_ID.nunique() == EXPECTED_PEOPLE,
        "Original_split_fingerprint_verified": computed_fp == EXPECTED_SPLIT_FINGERPRINT,
        "No_new_splits_generated": True,
        "M2_Version_A_RR_table_verified": observed_missing == EXPECTED_RR_MISSING and observed_rows_missing == EXPECTED_ROWS_WITH_MISSING_RR,
        "M0_reference_OOF_aligned": len(reference_oof) == EXPECTED_BEATS * N_REPEATS,
        "M2_Version_A_OOF_aligned": len(m2_oof) == EXPECTED_BEATS * N_REPEATS,
        "Candidate_has_17_features": X.shape == (EXPECTED_BEATS, 17),
        "XGBoost_smoke_test_passed": _smoke_probs.shape == (10, len(CLASS_ORDER)),
        "Unweighted_XGB_OOF_complete": len(xgb_predictions[UNWEIGHTED_NAME]) == EXPECTED_BEATS * N_REPEATS,
        "Weighted_XGB_OOF_complete": len(xgb_predictions[WEIGHTED_NAME]) == EXPECTED_BEATS * N_REPEATS,
        "Unweighted_fold_audit_complete": len(xgb_fold_audits[UNWEIGHTED_NAME]) == N_SPLITS * N_REPEATS,
        "Weighted_fold_audit_complete": len(xgb_fold_audits[WEIGHTED_NAME]) == N_SPLITS * N_REPEATS,
    }
    check_table = pd.DataFrame([
        {"Check": name, "Result": "PASS" if passed else "FAIL"}
        for name, passed in checks.items()
    ])
    display(check_table)
    check_table.to_csv(OUTPUT_DIR / "m4_xgb_final_validation_checks.csv", index=False)
    if not all(checks.values()):
        raise AssertionError("M4 XGBoost final validation failed. STOP.")

    run_manifest = {
        "phase": "M4 — XGBoost class weighting",
        "baseline": BASELINE_NAME,
        "experiments": {
            UNWEIGHTED_NAME: {"sample_weight": None},
            WEIGHTED_NAME: {"sample_weight": "balanced per training fold"},
        },
        "balanced_weight_definition": "n_train / (n_present_classes * count_of_class_in_training_fold)",
        "xgboost_parameters": XGB_PARAMS,
        "num_boost_round": NUM_BOOST_ROUND,
        "feature_count": int(X.shape[1]),
        "feature_names": FEATURE_NAMES,
        "valid_beats": EXPECTED_BEATS,
        "split_fingerprint": computed_fp,
        "splits_regenerated": False,
        "m2_rr_table": str(rr_path),
        "m2_baseline_oof": str(m2_oof_path),
        "bootstrap_samples": BOOTSTRAP_SAMPLES,
        "bootstrap_seed": BOOTSTRAP_SEED,
        "bootstrap_unit": "Record_ID",
        "python": sys.version,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "xgboost": xgb.__version__,
        "checks": checks,
        "clinical_use": "Research experiment only; not a clinically validated diagnostic model.",
    }
    (OUTPUT_DIR / "m4_xgb_run_manifest.json").write_text(json.dumps(run_manifest, indent=2))
    print("\nM4 XGBOOST RUN COMPLETE — ALL FINAL CHECKS PASS")
    print("Output directory:", OUTPUT_DIR)
else:
    print("Final training checks are deferred until RUN_MODEL_TRAINING=True.")

,Check,Result
0,Corrected_dataset_selected,PASS
1,M1_positions_loaded,PASS
2,M1_waveforms_match_corrected_dataset,PASS
3,105047_valid_beats,PASS
4,Expected_class_counts,PASS
5,46_records_45_people,PASS
6,Original_split_fingerprint_verified,PASS
7,No_new_splits_generated,PASS
8,M2_Version_A_RR_table_verified,PASS
9,M0_reference_OOF_aligned,PASS



M4 XGBOOST RUN COMPLETE — ALL FINAL CHECKS PASS
Output directory: /kaggle/working/ECG_Project/09_M4_XGBOOST_CLASS_WEIGHTING
